**Take Home Exercise 2**

**Q1: What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.**

To answer this question, I used the `pit_stops` dataset because it contains the time spent in each pit stop for every driver in each race. First, I joined it with the `drivers` dataset so I could identify drivers by code or name, and with the `races` dataset so I could identify each race clearly.

Then, I grouped the data by race and driver to calculate the average pit stop time for each driver in each race. After that, I separately grouped the data by race only to calculate the fastest pit stop (minimum duration) and the slowest pit stop (maximum duration) in each race. Finally, I joined these summaries together so the result shows each driver's average pit stop time along with the fastest and slowest pit stop for that race.

In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/gr5069/raw/f1_data"

pit_stops = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/pit_stops.csv")
)

drivers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/drivers.csv")
)

races = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/races.csv")
)

In [0]:
pit_stops.printSchema()
drivers.printSchema()
races.printSchema()

In [0]:
avg_driver_pit = (
    pit_stops
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("milliseconds").alias("avg_pit_ms")
    )
    .withColumn(
        "avg_pit_seconds",
        F.round(F.col("avg_pit_ms") / 1000, 3)
    )
)

In [0]:
race_fast_slow = (
    pit_stops
    .groupBy("raceId")
    .agg(
        F.min("milliseconds").alias("fastest_pit_ms"),
        F.max("milliseconds").alias("slowest_pit_ms")
    )
    .withColumn("fastest_pit_seconds", F.round(F.col("fastest_pit_ms") / 1000, 3))
    .withColumn("slowest_pit_seconds", F.round(F.col("slowest_pit_ms") / 1000, 3))
)

In [0]:
result_q1 = (
    avg_driver_pit
    .join(drivers.select("driverId", "code", "forename", "surname"), on="driverId", how="left")
    .join(races.select("raceId", "year", "name"), on="raceId", how="left")
    .join(race_fast_slow, on="raceId", how="left")
    .select(
        "year",
        F.col("name").alias("race_name"),
        "driverId",
        "code",
        "forename",
        "surname",
        "avg_pit_ms",
        "avg_pit_seconds",
        "fastest_pit_ms",
        "fastest_pit_seconds",
        "slowest_pit_ms",
        "slowest_pit_seconds"
    )
    .orderBy("year", "race_name", "avg_pit_seconds")
)

display(result_q1)

I first loaded the `pit_stops`, `drivers`, and `races` datasets using `spark.read.csv()` with headers and inferred schemas. The `pit_stops` dataset contains the pit stop duration for each driver in each race.

Next, I used `groupBy("raceId", "driverId")` and `avg("milliseconds")` to calculate the average pit stop time for each driver in each race. Then, I used `groupBy("raceId")` with `min("milliseconds")` and `max("milliseconds")` to find the fastest and slowest pit stop in each race.

After that, I joined the aggregated results with the `drivers` and `races` tables using `join()` so that the output includes driver names, driver codes, race names, and years. Finally, I used `select()` to keep only the relevant columns and `orderBy()` to sort the results in a readable order.


An alternative way to answer this question would be to use SQL instead of PySpark DataFrame operations. For example, I could create temporary views for the datasets and then use SQL `GROUP BY`, `AVG`, `MIN`, and `MAX` to produce the same result. This approach may be easier to read for users who are more familiar with SQL syntax.

**Q2: Rank order by finishing position the average time spent at the pit stop in each race.**

To answer this question, I first used the `pit_stops` dataset to calculate the average pit stop time for each driver in each race. Then, I joined that result with the `results` dataset because finishing position is recorded there.

After that, I grouped the data by race and finishing position to calculate the average pit stop time associated with each finishing position in each race. Finally, I joined the race information from the `races` dataset and sorted the output by race and finishing position so the results are shown in rank order.

In [0]:
results = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{base_path}/results.csv")
)

In [0]:
results.printSchema()

In [0]:
q2_base = (
    avg_driver_pit
    .join(
        results.select("raceId", "driverId", "positionOrder"),
        on=["raceId", "driverId"],
        how="inner"
    )
)

In [0]:
result_q2 = (
    q2_base
    .groupBy("raceId", "positionOrder")
    .agg(
        F.avg("avg_pit_ms").alias("avg_pit_ms_by_finish"),
        F.avg("avg_pit_seconds").alias("avg_pit_seconds_by_finish")
    )
    .join(races.select("raceId", "year", "name"), on="raceId", how="left")
    .withColumn("avg_pit_seconds_by_finish", F.round(F.col("avg_pit_seconds_by_finish"), 3))
    .select(
        "year",
        F.col("name").alias("race_name"),
        F.col("positionOrder").alias("finishing_position"),
        "avg_pit_ms_by_finish",
        "avg_pit_seconds_by_finish"
    )
    .orderBy("year", "race_name", "finishing_position")
)

display(result_q2)


I used the driver-level pit stop summary from Question 1, which already contains the average pit stop time for each driver in each race. Then, I joined that table with the `results` dataset using `raceId` and `driverId` so I could add each driver’s finishing position.

Next, I grouped the joined data by `raceId` and `positionOrder` and used `avg()` to calculate the average pit stop time for each finishing position within each race. After that, I joined the `races` dataset to include the race year and race name. Finally, I used `select()` to organize the final columns and `orderBy()` to sort the results by race and finishing position.

**Q3: Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.**

In [0]:
drivers.select("code").distinct().orderBy("code").show(200, truncate=False)

In [0]:
drivers_missing_code = drivers.filter(
    F.col("code").isNull() |
    (F.trim(F.col("code")) == "") |
    (F.col("code") == r"\N") |
    (F.lower(F.trim(F.col("code"))) == "null") |
    (F.lower(F.trim(F.col("code"))) == "na") |
    (F.lower(F.trim(F.col("code"))) == "none")
)

display(drivers_missing_code)

In [0]:
To answer this question, I first checked the `code` column in the `drivers` dataset and found that missing values are stored as `\N` rather than as nulls. I therefore treated `\N` as a missing code.

Then, I created a rule-based replacement for the missing values. My logic is to generate a three-letter code using the first three letters of the driver’s surname in uppercase. If the surname has fewer than three letters, I use the first three letters of the forename instead. Finally, I created a new column that keeps the original code when it exists and fills in the generated code only when the original code is missing.

In [0]:
drivers_q3 = (
    drivers
    .withColumn(
        "generated_code",
        F.when(
            F.length(F.col("surname")) >= 3,
            F.upper(F.substring(F.col("surname"), 1, 3))
        ).otherwise(
            F.upper(F.substring(F.col("forename"), 1, 3))
        )
    )
    .withColumn(
        "filled_code",
        F.when(
            F.col("code").isNull() |
            (F.trim(F.col("code")) == "") |
            (F.col("code") == r"\N"),
            F.col("generated_code")
        ).otherwise(F.col("code"))
    )
)

missing_filled = drivers_q3.filter(
    F.col("code").isNull() |
    (F.trim(F.col("code")) == "") |
    (F.col("code") == r"\N")
)

display(
    missing_filled.select(
        "driverId",
        "forename",
        "surname",
        "code",
        "generated_code",
        "filled_code"
    )
)

In [0]:
drivers_final_q3 = (
    drivers_q3
    .drop("code")
    .withColumnRenamed("filled_code", "code")
    .drop("generated_code")
)

display(drivers_final_q3)


I first identified missing driver codes by checking whether `code` was null, blank, or equal to `\N`. In this dataset, `\N` is used as the placeholder for missing values, so I treated it as missing data.

Next, I used `withColumn()` and `when()` to create a generated replacement code. The replacement is based on the first three letters of the driver’s surname in uppercase using `substring()` and `upper()`. If the surname is shorter than three characters, the code instead uses the first three letters of the forename. Finally, I created `filled_code`, which preserves the original code when it exists and only inserts the generated value when the original code is missing.

**Q4: Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".**


To answer this question, I joined the `results`, `drivers`, and `races` datasets so that each driver in each race has both a date of birth and a race date. I define age as the driver’s completed age in years on the date of the race.

Then, I created a new column called `Age` and used it to compare drivers within each race. Finally, I identified the youngest driver as the one with the minimum age and the oldest driver as the one with the maximum age in each race.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

q4_base = (
    results
    .select("raceId", "driverId")
    .join(
        drivers.select("driverId", "forename", "surname", "dob"),
        on="driverId",
        how="left"
    )
    .join(
        races.select("raceId", "year", "name", "date"),
        on="raceId",
        how="left"
    )
    .withColumn("dob", F.to_date("dob"))
    .withColumn("race_date", F.to_date("date"))
)

In [0]:
q4_with_age = (
    q4_base
    .withColumn(
        "Age",
        F.floor(F.months_between(F.col("race_date"), F.col("dob")) / 12)
    )
)

display(q4_with_age.select("year", "name", "race_date", "forename", "surname", "dob", "Age"))

In [0]:
youngest_window = Window.partitionBy("raceId").orderBy(F.col("Age").asc(), F.col("driverId").asc())
oldest_window = Window.partitionBy("raceId").orderBy(F.col("Age").desc(), F.col("driverId").asc())

youngest_per_race = (
    q4_with_age
    .withColumn("rn", F.row_number().over(youngest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId",
        "year",
        F.col("name").alias("race_name"),
        "race_date",
        F.concat_ws(" ", "forename", "surname").alias("youngest_driver"),
        F.col("Age").alias("youngest_age")
    )
)

oldest_per_race = (
    q4_with_age
    .withColumn("rn", F.row_number().over(oldest_window))
    .filter(F.col("rn") == 1)
    .select(
        "raceId",
        F.concat_ws(" ", "forename", "surname").alias("oldest_driver"),
        F.col("Age").alias("oldest_age")
    )
)

result_q4 = (
    youngest_per_race
    .join(oldest_per_race, on="raceId", how="inner")
    .select(
        "year",
        "race_name",
        "race_date",
        "youngest_driver",
        "youngest_age",
        "oldest_driver",
        "oldest_age"
    )
    .orderBy("year", "race_name")
)

display(result_q4)

I first joined the `results`, `drivers`, and `races` datasets so that each row represents one driver in one race, with both date of birth and race date available. Then, I converted the `dob` and `date` columns into proper date format using `to_date()`.

Next, I created a new column called `Age` by calculating the number of months between the race date and the driver’s date of birth, dividing by 12, and applying `floor()` so that age is measured in full completed years. After that, I used window functions and `row_number()` to rank drivers within each race by age, once from youngest to oldest and once from oldest to youngest. Finally, I kept the top row from each ranking and joined the two results together.

### Definition of Age

I define age as the driver’s completed age in years on the date of the race. In other words, I count only full years that the driver had already completed by race day, rather than using fractional years. This definition is clear, consistent, and easy to interpret.

**Q5: At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver**

To answer this question, I used the `results` dataset because it contains each driver’s finishing position in each race. I joined it with the `races` dataset so that I could order races chronologically for each driver.

Then, I created three indicator columns: one for wins (1st place), one for second places, and one for third places. After that, I used window functions to calculate cumulative totals for each driver over time. This allows me to show, at any given race, how many wins, second places, and third places each driver had accumulated up to that point.

In [0]:
q5_base = (
    results
    .select("raceId", "driverId", "positionOrder")
    .join(
        races.select("raceId", "year", "name", "date"),
        on="raceId",
        how="left"
    )
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("race_date", F.to_date("date"))
)

In [0]:
q5_flags = (
    q5_base
    .withColumn("win_flag", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
    .withColumn("second_flag", F.when(F.col("positionOrder") == 2, 1).otherwise(0))
    .withColumn("third_flag", F.when(F.col("positionOrder") == 3, 1).otherwise(0))
)

display(q5_flags)

In [0]:
podium_window = (
    Window
    .partitionBy("driverId")
    .orderBy(F.col("race_date").asc(), F.col("raceId").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

In [0]:
result_q5 = (
    q5_flags
    .withColumn("wins", F.sum("win_flag").over(podium_window))
    .withColumn("second_places", F.sum("second_flag").over(podium_window))
    .withColumn("third_places", F.sum("third_flag").over(podium_window))
    .select(
        "year",
        F.col("name").alias("race_name"),
        "race_date",
        "driverId",
        F.concat_ws(" ", "forename", "surname").alias("driver_name"),
        F.col("positionOrder").alias("finishing_position"),
        "wins",
        "second_places",
        "third_places"
    )
    .orderBy("year", "race_date", "driver_name")
)

display(result_q5)


I first joined the `results`, `races`, and `drivers` datasets so that each row represents one driver in one race, along with the race date and finishing position. Then, I created three indicator columns: `win_flag`, `second_flag`, and `third_flag`. Each indicator equals 1 when the driver finished in that podium position and 0 otherwise.

Next, I used a window function partitioned by `driverId` and ordered by race date to calculate cumulative totals over time. I applied `sum()` over that window to count how many wins, second places, and third places each driver had accumulated by each race. Finally, I selected the relevant columns and sorted the results in chronological order.

**Q6: Which drivers have the most wins over time?**

To answer this, I used the `results` dataset and identified race wins as finishing in first place. Then, I joined the data with the `races` and `drivers` datasets so I could track wins by driver over time. After that, I used a cumulative sum to calculate running win totals for each driver across races. Finally, I created a summary table showing which drivers have the highest total number of wins.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

q6_base = (
    results
    .select("raceId", "driverId", "positionOrder")
    .join(
        races.select("raceId", "year", "name", "date"),
        on="raceId",
        how="left"
    )
    .join(
        drivers.select("driverId", "forename", "surname"),
        on="driverId",
        how="left"
    )
    .withColumn("race_date", F.to_date("date"))
)

In [0]:
q6_wins = (
    q6_base
    .withColumn("win_flag", F.when(F.col("positionOrder") == 1, 1).otherwise(0))
)

win_window = (
    Window
    .partitionBy("driverId")
    .orderBy(F.col("race_date").asc(), F.col("raceId").asc())
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

q6_running_wins = (
    q6_wins
    .withColumn("cumulative_wins", F.sum("win_flag").over(win_window))
    .select(
        "year",
        F.col("name").alias("race_name"),
        "race_date",
        "driverId",
        F.concat_ws(" ", "forename", "surname").alias("driver_name"),
        "positionOrder",
        "win_flag",
        "cumulative_wins"
    )
    .orderBy("race_date", "driver_name")
)

display(q6_running_wins)

In [0]:
q6_total_wins = (
    q6_wins
    .groupBy("driverId", "forename", "surname")
    .agg(F.sum("win_flag").alias("total_wins"))
    .withColumn("driver_name", F.concat_ws(" ", "forename", "surname"))
    .select("driverId", "driver_name", "total_wins")
    .orderBy(F.col("total_wins").desc(), F.col("driver_name").asc())
)

display(q6_total_wins)

In [0]:
#Top 10
display(q6_total_wins.limit(10))

I first joined the `results`, `races`, and `drivers` datasets so that each row contains a driver, a race, the race date, and the finishing position. Then, I created a `win_flag` column that equals 1 when a driver finished in first place and 0 otherwise.

Next, I used a window function partitioned by `driverId` and ordered by race date to calculate a running total of wins over time for each driver. This shows how each driver’s number of wins grows throughout their career. Finally, I grouped the data by driver and summed the `win_flag` column to create a ranking table of total career wins.